# 03 · Feature Engineering

**Netflix Customer Intelligence, Churn Prediction & AI Analytics Copilot** · Week 2, Days 1–2

This notebook turns the nine database tables into **one row per customer** with 43 features (PRD Section 9.2), and creates the training table for the models in the next notebooks (the CLV model in notebook 06 uses lifetime features instead).

### The key design choice: describe each customer *as of a date*
In this dataset a customer who cancels stops watching, stops paying and loses their subscription. If features are measured at the end of the data, they simply reveal who already left (recency, plan, payments), and a model scores ~0.99 AUC without learning anything useful. Notebook 04 demonstrates this explicitly.

So every feature here is computed **as of a cutoff date**, using only rows dated on or before it:

| Snapshot | Date | Used for | Label |
|---|---|---|---|
| **Training snapshot** | 2025-12-31 (6 months of viewing history) | churn model and segmentation | cancels in the next ~6 months (to 2026-06-30) |
| **Current snapshot** | 2026-06-30 (end of the data) | scoring today's customers (Week 3 copilot) | none |

A "time-travel test" below proves the features use nothing from after the cutoff.

| Step | Output |
|---|---|
| 1–3 | Load the tables, set the dates, run the leak-free test |
| 4–5 | Build both snapshots and the feature dictionary |
| 6–10 | Check the features: missing values, signal strength, relationships, correlations, timeline |
| 11 | Save `customer_features_train.csv` and `customer_features_current.csv` |

In [ ]:
PREFIX = "fe"

import sys, warnings, json
from pathlib import Path
warnings.filterwarnings("ignore")

# Find the project root (the folder that contains src/utils/config.py)
ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "src" / "utils" / "config.py").exists()), None)
if ROOT is None:                          # fallback: an environment variable set before starting Jupyter
    import os
    _env = os.environ.get("NETFLIX_PROJECT_ROOT", "")
    if _env and (Path(_env) / "src" / "utils" / "config.py").exists():
        ROOT = Path(_env).resolve()
assert ROOT is not None, (f"This notebook is running in {Path.cwd()}, which is not inside the project folder. "
                          "Open the copy in <project>/notebooks (start Jupyter from the project root), "
                          "or set the NETFLIX_PROJECT_ROOT environment variable.")
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

PROCESSED = ROOT / "data" / "processed"
MODELS    = ROOT / "models"
REPORTS   = ROOT / "reports"
FIG_DIR   = REPORTS / "figures"
for d in (PROCESSED, MODELS, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

NETFLIX_RED, DARK, GREY, BLUE = "#E50914", "#221F1F", "#8C8C8C", "#3B6EA8"
sns.set_theme(style="whitegrid", context="notebook",
              rc={"axes.spines.top": False, "axes.spines.right": False})
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "axes.titleweight": "bold", "axes.titlesize": 13})


def finish(num, name, fig=None):
    """Tidy the figure, save it to reports/figures and show it."""
    fig = fig or plt.gcf()
    fig.tight_layout()
    path = FIG_DIR / f"{PREFIX}_{num:02d}_{name}.png"
    fig.savefig(path, bbox_inches="tight")
    plt.show()
    print(f"Saved chart: {path.relative_to(ROOT)}")

print("Project root:", ROOT)

In [ ]:
# ── 1. Load the cleaned tables from MySQL ────────────────────────────────────
from src.database.connection import get_engine
from src.features.load_tables import load_clean_tables
from src.features import build_features as bf

tables = load_clean_tables(get_engine())
for t, df in tables.items():
    print(f"{t:20s} {df.shape[0]:>8,} rows x {df.shape[1]:>2} cols")

In [ ]:
# ── 2. The dates that define the problem ─────────────────────────────────────
AS_OF        = tables["viewing_activity"].viewing_date.max()        # end of the data
TRAIN_CUTOFF = pd.Timestamp("2025-12-31")                           # features are measured here
HORIZON_DAYS = (AS_OF - TRAIN_CUTOFF).days                          # churn label window: cutoff -> end of data

print(f"Training cutoff : {TRAIN_CUTOFF.date()}")
print(f"Churn label     : cancelled between {(TRAIN_CUTOFF + pd.Timedelta(days=1)).date()} and {AS_OF.date()} ({HORIZON_DAYS} days)")
print(f"Current snapshot: {AS_OF.date()}")

In [ ]:
# ── 3. Time-travel test: do the features use anything from after the cutoff? ──
# Rebuild the snapshot from data that STOPS at the cutoff. If the result is identical to the one built
# from the full data, no feature can have used the future.
leak_free = bf.snapshot_is_leak_free(tables, TRAIN_CUTOFF)
print("Features identical when future rows are removed:", leak_free)
assert leak_free, "A feature is using data from after the cutoff - fix build_features.py"

In [ ]:
# ── 4. Build the two snapshots and the labels ────────────────────────────────
X_train = bf.build_snapshot(tables, TRAIN_CUTOFF, population="active")   # active customers on the cutoff date
y_churn = bf.churn_label(tables, X_train.index, TRAIN_CUTOFF, HORIZON_DAYS).rename("churn_next_6m")
X_now   = bf.build_snapshot(tables, AS_OF, population="active")           # active customers today

print(f"Training snapshot : {X_train.shape[0]:,} customers x {X_train.shape[1]} features")
print(f"  churned within the label window : {int(y_churn.sum()):,} ({y_churn.mean():.1%})")
print(f"Current snapshot  : {X_now.shape[0]:,} customers x {X_now.shape[1]} features")
X_train.head()

In [ ]:
# ── 5. Feature dictionary ────────────────────────────────────────────────────
dictionary = pd.DataFrame([{"feature": f, "group": g, "source": s, "description": d}
                           for f, (g, s, d) in bf.FEATURE_META.items()])
dictionary["type"] = np.where(dictionary.feature.isin(bf.CATEGORICAL), "categorical", "numeric")
dictionary.to_csv(REPORTS / "feature_dictionary.csv", index=False)

print(dictionary.groupby(["group", "source"]).size().unstack(fill_value=0))
print(f"\n{(dictionary.source == 'PRD').sum()} features from PRD Section 9.2 + "
      f"{(dictionary.source == 'EDA').sum()} additional features motivated by the EDA (the PRD asks for at least 3)")
print("Still to come: churn_risk_score (added by the churn model in notebook 04)")
dictionary

## Checking the features

In [ ]:
# Chart 1 — Missing values (a gap usually means "the customer never did this")
miss = X_train.isna().mean().loc[lambda s: s > 0].sort_values()
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(miss.index, miss.values * 100, color=NETFLIX_RED)
for y, v in enumerate(miss.values):
    ax.text(v * 100 + 1, y, f"{v:.0%}", va="center", fontsize=9)
ax.set_xlim(0, 100); ax.set_xlabel("% of customers with no value")
ax.set_title("Features with missing values (training snapshot)")
finish(1, "missing_values", fig)
print("Missing values are kept as NaN and handled inside the model pipelines (median imputation + a 'was missing' flag).")

In [ ]:
# Chart 2 — How much signal does each feature carry on its own? (univariate ROC-AUC against churn)
from sklearn.metrics import roc_auc_score
aucs = {}
for f in bf.NUMERIC:
    s = X_train[f].fillna(X_train[f].median())
    if s.nunique() > 1:
        aucs[f] = roc_auc_score(y_churn, s)
sig = pd.Series(aucs)
top = (sig - 0.5).abs().sort_values(ascending=False).head(15).index
plot = sig.loc[top].sort_values()
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.barh(plot.index, plot.values - 0.5, left=0.5, color=[NETFLIX_RED if v > 0.5 else BLUE for v in plot.values])
ax.axvline(0.5, color=DARK, lw=1)
ax.set_xlabel("Single-feature ROC-AUC (0.5 = no signal)")
ax.set_title("Strongest individual signals for churn in the next 6 months")
ax.text(0.99, 0.02, "red = higher value, more churn\nblue = higher value, less churn", transform=ax.transAxes,
        ha="right", va="bottom", fontsize=9, bbox=dict(boxstyle="round", fc="white", ec=GREY))
finish(2, "feature_signal", fig)
print(f"Strongest single feature: {sig.sub(0.5).abs().idxmax()} (AUC {sig[sig.sub(0.5).abs().idxmax()]:.3f}). "
      "No feature is above ~0.6, so churn here is noisy and we need many weak signals combined.")

In [ ]:
# Chart 3 — Churn rate across the range of six important features
key = ["customer_tenure_days", "failed_payment_rate", "customer_lifetime_value",
       "avg_customer_satisfaction", "support_ticket_count", "engagement_score"]
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
base = y_churn.mean()
for ax, f in zip(axes.ravel(), key):
    s = X_train[f]
    try:
        bins = pd.qcut(s, 5, duplicates="drop")
    except ValueError:
        bins = pd.cut(s, 5)
    g = y_churn.groupby(bins, observed=True).agg(["mean", "size"])
    fmt = (lambda v: f"{v:,.0f}") if s.max() >= 100 else (lambda v: f"{v:.2f}".rstrip("0").rstrip("."))
    labels = [f"{fmt(max(iv.left, 0))}-{fmt(iv.right)}" for iv in g.index]
    ax.bar(range(len(g)), g["mean"] * 100, color=NETFLIX_RED)
    ax.axhline(base * 100, color=DARK, ls="--", lw=1)
    ax.set_xticks(range(len(g))); ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
    ax.set_title(f, fontsize=10); ax.set_ylabel("Churn rate (%)"); ax.set_xlabel("feature range (low to high)", fontsize=8)
fig.suptitle(f"Churn rate by feature range (dashed = overall {base:.1%})", fontweight="bold")
finish(3, "churn_by_feature_range", fig)

In [ ]:
# Chart 4 — Which features carry the same information? (correlation of the numeric features)
cols = ["customer_tenure_days", "monthly_price", "customer_lifetime_value", "payments_total", "failed_payment_rate",
        "failed_payments_90d", "watch_min_90d", "sessions_90d", "engagement_score", "days_since_last_activity",
        "monthly_watch_time_trend", "avg_watch_time_per_session", "avg_completion_pct", "support_ticket_count",
        "billing_ticket_count", "open_ticket_count", "avg_customer_satisfaction", "feedback_count", "avg_rating"]
corr = X_train[cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, mask=mask, cmap="RdBu_r", center=0, vmin=-1, vmax=1, annot=True, fmt=".1f",
            annot_kws={"size": 7}, cbar_kws={"label": "Correlation"}, ax=ax)
ax.set_title("Correlation between key features")
finish(4, "feature_correlations", fig)
pairs = (corr.where(~mask & (corr.abs() < 1)).stack().abs().sort_values(ascending=False).head(3))
print("Most overlapping pairs:", ", ".join(f"{a} / {b} ({v:.2f})" for (a, b), v in pairs.items()))

In [ ]:
# Chart 5 — The snapshot design on a timeline
import matplotlib.dates as mdates
v0 = tables["viewing_activity"].viewing_date.min()
left, right = pd.Timestamp("2025-05-01"), pd.Timestamp("2026-08-15")
fig, ax = plt.subplots(figsize=(11, 3.8))
def bar(y, start, end, color, label, fc="white"):
    ax.barh(y, (end - start).days, left=start, height=0.55, color=color)
    ax.text(start + (end - start) / 2, y, label, ha="center", va="center", color=fc, fontsize=8.5, fontweight="bold")
bar(3, v0, TRAIN_CUTOFF, GREY, "Features: 6 months of viewing")
bar(3, TRAIN_CUTOFF + pd.Timedelta(days=1), AS_OF, NETFLIX_RED, "Label: did they cancel?")
bar(2, left, TRAIN_CUTOFF, GREY, "Payments, tickets, feedback: all history to the cutoff (from 2019)")
bar(1, v0, AS_OF, DARK, "Today's snapshot (features as of 2026-06-30, used for scoring)")
ax.axvline(TRAIN_CUTOFF, color="black", ls="--", lw=1.2)
ax.text(TRAIN_CUTOFF, 3.45, " training cutoff 2025-12-31", fontsize=9, va="bottom")
ax.set_xlim(left, right); ax.set_ylim(0.5, 3.9); ax.set_yticks([])
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2)); ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.set_title("What the model sees, and what it must predict"); ax.grid(False, axis="y")
finish(5, "snapshot_timeline", fig)

**What the checks show**
- **Missing values mean "never did this", not errors.** 63% of customers have no resolved ticket (so no satisfaction score or resolution time), 62% never left a review, 54% never raised a ticket, 44% never had a failed payment, and 8% had no viewing session at all. The model pipelines keep a "was missing" flag for these.
- **Failed payments are the clearest warning sign.** 56% of customers have at least one; they went on to churn at 20.4% against 15.0% for those with none. The 10% of customers with a failure rate above 13% churned at 30.3%, about double the 15.7% of the lowest group.
- **Newer, less satisfied and heavier-support customers churn more.** The newest fifth of customers churned at 23.0% against 14.7% for the fourth fifth; customers with an average CSAT of 2 or below at 24.0% against 13.1% for 4–5; customers with 2+ tickets at 23.2% against 17.4%; auto-renew off at 21.2% against 17.3% for on.
- **Viewing *before* the cutoff is not a strong early warning.** The engagement score is almost unrelated to later cancellation (15.9%–20.7% across fifths, slightly higher for the most engaged). The strong engagement pattern in the EDA (chart 20) appears only when viewing is measured right up to the cancellation date, because people stop watching once they leave. Notebook 04 demonstrates this with an experiment.
- **Every single feature is weak** (the best has a ROC-AUC of only 0.571), so a model has to combine many weak signals and honest accuracy will be modest.
- **Some features overlap:** sessions, watch minutes and engagement score correlate at 0.87-0.91, and monthly price and lifetime value at 0.8. The models below cope with this (regularisation for logistic regression, trees for the rest).
- **The time-travel test passed:** removing every row dated after the cutoff leaves the features unchanged, so nothing from the future is used.

**Next:** notebook 04 trains and evaluates the churn model on `customer_features_train.csv`.

In [ ]:
# ── 11. Save the modelling tables ────────────────────────────────────────────
train_out = X_train.join(y_churn)
train_out.to_csv(PROCESSED / "customer_features_train.csv")
X_now.to_csv(PROCESSED / "customer_features_current.csv")
meta = {"train_cutoff": str(TRAIN_CUTOFF.date()), "as_of": str(AS_OF.date()), "horizon_days": int(HORIZON_DAYS), "n_train": int(len(X_train)), "n_current": int(len(X_now)),
        "churn_rate_train": float(y_churn.mean()), "n_features": len(bf.FEATURES)}
json.dump(meta, open(PROCESSED / "feature_meta.json", "w"), indent=2)

print("Saved:")
for p in ["customer_features_train.csv", "customer_features_current.csv", "feature_meta.json"]:
    print("  data/processed/" + p)
print("  reports/feature_dictionary.csv")
print(meta)